# theoretical price of a call option
Using bsm, heston, merton, kou model to price a call option and put option

In [2]:
# Semi-analytic Heston uses QuantLib (import QuantLib as ql).
# pip install QuantLib


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
# -----------------------------------------------------------------------
# Replication of Tsay textbook Examples 6.6 and 6.8
# (Analysis of Financial Time Series, Tsay 2010, Chapter 6)
#
# Example 6.6 (Black-Scholes, no jumps):
#   S=80, K=81, T=0.25, r=0.08, sigma=0.2  =>  call=$3.49
#
# Example 6.8 (Kou double-exponential jump-diffusion):
#   same parameters + lambda=10, kappa=-0.02, eta=0.02  =>  call=$3.92, put=$3.31
#
# Tsay parameterisation: log-jump Y ~ Laplace(kappa, eta)
#   mean(Y) = kappa,  Var(Y) = 2*eta^2
#   psi = E[J]-1 = exp(kappa)/(1-eta^2) - 1  (compensator)
# -----------------------------------------------------------------------

# Parameters (shared)
S      = 80      # current stock price
K      = 81      # strike price
T      = 0.25    # time to maturity (years)
r      = 0.08    # risk-free rate
sigma  = 0.2     # diffusion volatility
N      = 50      # number of Merton series terms, which is the number of jumps

# Heston parameters 
v0 = 0.04 # initial variance
kappa = 0.1 # mean reversion speed
theta = 0.04 # long-run variance
sigma_v = 0.1 # vol-of-vol
rho = 0.7 # correlation between spot and variance

# Merton jump-diffusion parameters (lognormal jumps)
Lambda = 10      # jump intensity (jumps per year)
muJ    = -0.02   # mean of log-jump
sigmaJ = 0.02    # std dev of log-jump

# Kou jump-diffusion parameters (double-exponential / Laplace jumps, Tsay notation)
lam_kou   = 10     # jump intensity
kappa_kou = -0.02  # mean of log-jump  (Laplace location parameter)
eta_kou   = 0.02   # scale of log-jump (Laplace scale parameter, must be < 1)

# ---- Black-Scholes (no jumps) ----
from merton_jump_pricing import black_scholes_call, merton_jump_pricing_call, merton_jump_pricing_put
from Heston_EulerAndMilstein import heston_european_mc, heston_quantlib_vanilla

bs_c     = black_scholes_call(S, K, T, r, sigma)
bs_p     = bs_c - S + K * np.exp(-r * T)

# ---- Heston (Euler MC paths + QuantLib semi-analytic) ----
HESTON_MC_PATHS = 100_000
HESTON_MC_STEPS = 252
HESTON_MC_SEED = 42

heston_mc_c = heston_european_mc(
    S,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma_v,
    rho,
    is_call=True,
    NoOfPaths=HESTON_MC_PATHS,
    NoOfSteps=HESTON_MC_STEPS,
    seed=HESTON_MC_SEED,
)
heston_mc_p = heston_european_mc(
    S,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma_v,
    rho,
    is_call=False,
    NoOfPaths=HESTON_MC_PATHS,
    NoOfSteps=HESTON_MC_STEPS,
    seed=HESTON_MC_SEED,
)

heston_ql_c = heston_quantlib_vanilla(
    S, K, T, r, v0, kappa, theta, sigma_v, rho, q=0.0, is_call=True
)
heston_ql_p = heston_quantlib_vanilla(
    S, K, T, r, v0, kappa, theta, sigma_v, rho, q=0.0, is_call=False
)

# ---- Merton (lognormal jumps) ----
merton_c = merton_jump_pricing_call(S, K, T, r, sigma, Lambda, muJ, sigmaJ, N)
merton_p = merton_jump_pricing_put( S, K, T, r, sigma, Lambda, muJ, sigmaJ, N)

# ---- Kou (double-exponential / Laplace jumps, Tsay parameterisation) ----
from kou_jump_pricing import kou_jump_call_tsay, kou_jump_put_tsay
kou_c = kou_jump_call_tsay(S, K, T, r, sigma, lam_kou, kappa_kou, eta_kou)
kou_p = kou_jump_put_tsay( S, K, T, r, sigma, lam_kou, kappa_kou, eta_kou)

# ---- Print results ----
print("=" * 55)
print(f"{'Model':<30} {'Out-of-the-money Call':>10} {'In-the-money Put':>10}")
print("-" * 55)
print("Parameters:")
print(f"S = {S}, K = {K}, T = {T}, r = {r}, sigma = {sigma}")
print(
    f"Heston: v0 = {v0}, kappa = {kappa}, theta = {theta}, "
    f"sigma_v = {sigma_v}, rho = {rho}"
)
print(
    f"Heston MC: paths = {HESTON_MC_PATHS}, steps = {HESTON_MC_STEPS}, seed = {HESTON_MC_SEED}"
)
print(
    f"Merton: Lambda = {Lambda}, muJ = {muJ}, sigmaJ = {sigmaJ}, "
    f"N (series terms) = {N}"
)
print(
    f"Kou (Tsay): lambda = {lam_kou}, kappa = {kappa_kou}, eta = {eta_kou}"
)
print("-" * 55)
print(f"{'Black-Scholes (no jumps)':<30} {'${:.2f}'.format(bs_c):>10} {'${:.2f}'.format(bs_p):>10}")
print(f"{'Heston (Euler MC)':<30} {'${:.2f}'.format(heston_mc_c):>10} {'${:.2f}'.format(heston_mc_p):>10}")
print(f"{'Heston (QuantLib analytic)':<30} {'${:.2f}'.format(heston_ql_c):>10} {'${:.2f}'.format(heston_ql_p):>10}")
print(f"{'Merton jump-diffusion':<30} {'${:.2f}'.format(merton_c):>10} {'${:.2f}'.format(merton_p):>10}")
print(f"{'Kou jump-diffusion':<30} {'${:.2f}'.format(kou_c):>10} {'${:.2f}'.format(kou_p):>10}")
print("-" * 55)
print(
    "Heston Euler MC − QuantLib (semi-analytic):  "
    f"Δcall = {heston_mc_c - heston_ql_c:+.4f},  Δput = {heston_mc_p - heston_ql_p:+.4f}"
)
print("=" * 55)

Model                                Call        Put
-------------------------------------------------------
Parameters:
S = 80, K = 81, T = 0.25, r = 0.08, sigma = 0.2
Heston: v0 = 0.04, kappa = 0.1, theta = 0.04, sigma_v = 0.1, rho = 0.7
Heston MC: paths = 100000, steps = 252, seed = 42
Lambda = 10, muJ = -0.02, sigmaJ = 0.02, N = 50
-------------------------------------------------------
Black-Scholes (no jumps)            $3.49      $2.89
Heston (Euler MC)                   $3.46      $2.86
Heston (QuantLib analytic)          $3.47      $2.87
Merton jump-diffusion               $3.78      $3.18
Kou jump-diffusion                  $3.92      $3.31
-------------------------------------------------------
Heston Euler MC − QuantLib (semi-analytic):  Δcall = -0.0163,  Δput = -0.0088


In [ ]:
# in-the-money for call option pricing
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
# -----------------------------------------------------------------------
# Replication of Tsay textbook Examples 6.6 and 6.8
# (Analysis of Financial Time Series, Tsay 2010, Chapter 6)
#
# Example 6.6 (Black-Scholes, no jumps): (change S=80->82)
#   S=82, K=81, T=0.25, r=0.08, sigma=0.2  =>  call=$3.49
#
# Example 6.8 (Kou double-exponential jump-diffusion):
#   same parameters + lambda=10, kappa=-0.02, eta=0.02  =>  call=$3.92, put=$3.31
#
# Tsay parameterisation: log-jump Y ~ Laplace(kappa, eta)
#   mean(Y) = kappa,  Var(Y) = 2*eta^2
#   psi = E[J]-1 = exp(kappa)/(1-eta^2) - 1  (compensator)
# -----------------------------------------------------------------------

# Parameters (shared)
S      = 82      # current stock price
K      = 81      # strike price
T      = 0.25    # time to maturity (years)
r      = 0.08    # risk-free rate
sigma  = 0.2     # diffusion volatility
N      = 50      # number of Merton series terms, which is the number of jumps

# Heston parameters 
v0 = 0.04 # initial variance
kappa = 0.1 # mean reversion speed
theta = 0.04 # long-run variance
sigma_v = 0.1 # vol-of-vol
rho = 0.7 # correlation between spot and variance

# Merton jump-diffusion parameters (lognormal jumps)
Lambda = 10      # jump intensity (jumps per year)
muJ    = -0.02   # mean of log-jump
sigmaJ = 0.02    # std dev of log-jump

# Kou jump-diffusion parameters (double-exponential / Laplace jumps, Tsay notation)
lam_kou   = 10     # jump intensity
kappa_kou = -0.02  # mean of log-jump  (Laplace location parameter)
eta_kou   = 0.02   # scale of log-jump (Laplace scale parameter, must be < 1)

# ---- Black-Scholes (no jumps) ----
from merton_jump_pricing import black_scholes_call, merton_jump_pricing_call, merton_jump_pricing_put
from Heston_EulerAndMilstein import heston_european_mc, heston_quantlib_vanilla

bs_c     = black_scholes_call(S, K, T, r, sigma)
bs_p     = bs_c - S + K * np.exp(-r * T)

# ---- Heston (Euler MC paths + QuantLib semi-analytic) ----
HESTON_MC_PATHS = 100_000
HESTON_MC_STEPS = 252
HESTON_MC_SEED = 42

heston_mc_c = heston_european_mc(
    S,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma_v,
    rho,
    is_call=True,
    NoOfPaths=HESTON_MC_PATHS,
    NoOfSteps=HESTON_MC_STEPS,
    seed=HESTON_MC_SEED,
)
heston_mc_p = heston_european_mc(
    S,
    K,
    T,
    r,
    v0,
    kappa,
    theta,
    sigma_v,
    rho,
    is_call=False,
    NoOfPaths=HESTON_MC_PATHS,
    NoOfSteps=HESTON_MC_STEPS,
    seed=HESTON_MC_SEED,
)

heston_ql_c = heston_quantlib_vanilla(
    S, K, T, r, v0, kappa, theta, sigma_v, rho, q=0.0, is_call=True
)
heston_ql_p = heston_quantlib_vanilla(
    S, K, T, r, v0, kappa, theta, sigma_v, rho, q=0.0, is_call=False
)

# ---- Merton (lognormal jumps) ----
merton_c = merton_jump_pricing_call(S, K, T, r, sigma, Lambda, muJ, sigmaJ, N)
merton_p = merton_jump_pricing_put( S, K, T, r, sigma, Lambda, muJ, sigmaJ, N)

# ---- Kou (double-exponential / Laplace jumps, Tsay parameterisation) ----
from kou_jump_pricing import kou_jump_call_tsay, kou_jump_put_tsay
kou_c = kou_jump_call_tsay(S, K, T, r, sigma, lam_kou, kappa_kou, eta_kou)
kou_p = kou_jump_put_tsay( S, K, T, r, sigma, lam_kou, kappa_kou, eta_kou)

# ---- Print results ----
print("=" * 55)
print(f"{'Model':<30} {'In-the-money Call':>10} {'Out-of-the-money Put':>10}")
print("-" * 55)
print("Parameters:")
print(f"S = {S}, K = {K}, T = {T}, r = {r}, sigma = {sigma}")
print(
    f"Heston: v0 = {v0}, kappa = {kappa}, theta = {theta}, "
    f"sigma_v = {sigma_v}, rho = {rho}"
)
print(
    f"Heston MC: paths = {HESTON_MC_PATHS}, steps = {HESTON_MC_STEPS}, seed = {HESTON_MC_SEED}"
)
print(
    f"Merton: Lambda = {Lambda}, muJ = {muJ}, sigmaJ = {sigmaJ}, "
    f"N (series terms) = {N}"
)
print(
    f"Kou (Tsay): lambda = {lam_kou}, kappa = {kappa_kou}, eta = {eta_kou}"
)
print("-" * 55)
print(f"{'Black-Scholes (no jumps)':<30} {'${:.2f}'.format(bs_c):>10} {'${:.2f}'.format(bs_p):>10}")
print(f"{'Heston (Euler MC)':<30} {'${:.2f}'.format(heston_mc_c):>10} {'${:.2f}'.format(heston_mc_p):>10}")
print(f"{'Heston (QuantLib analytic)':<30} {'${:.2f}'.format(heston_ql_c):>10} {'${:.2f}'.format(heston_ql_p):>10}")
print(f"{'Merton jump-diffusion':<30} {'${:.2f}'.format(merton_c):>10} {'${:.2f}'.format(merton_p):>10}")
print(f"{'Kou jump-diffusion':<30} {'${:.2f}'.format(kou_c):>10} {'${:.2f}'.format(kou_p):>10}")
print("-" * 55)
print(
    "Heston Euler MC − QuantLib (semi-analytic):  "
    f"Δcall = {heston_mc_c - heston_ql_c:+.4f},  Δput = {heston_mc_p - heston_ql_p:+.4f}"
)
print("=" * 55)

Model                                Call        Put
-------------------------------------------------------
Parameters:
S = 82, K = 81, T = 0.25, r = 0.08, sigma = 0.2
Heston: v0 = 0.04, kappa = 0.1, theta = 0.04, sigma_v = 0.1, rho = 0.7
Heston MC: paths = 100000, steps = 252, seed = 42
Merton: Lambda = 10, muJ = -0.02, sigmaJ = 0.02, N (series terms) = 50
Kou (Tsay): lambda = 10, kappa = -0.02, eta = 0.02
-------------------------------------------------------
Black-Scholes (no jumps)            $4.69      $2.08
Heston (Euler MC)                   $4.62      $2.03
Heston (QuantLib analytic)          $4.64      $2.04
Merton jump-diffusion               $4.98      $2.38
Kou jump-diffusion                  $5.11      $2.51
-------------------------------------------------------
Heston Euler MC − QuantLib (semi-analytic):  Δcall = -0.0139,  Δput = -0.0061
